In [2]:
import os
import time
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

In [4]:
# ---- 1. Stratified sample of 200 pairs ----
pairs = pd.read_csv('../data/processed/abt_buy_pairs.csv')

positives = pairs[pairs['label'] == 1].sample(100, random_state=7)
negatives = pairs[pairs['label'] == 0].sample(100, random_state=7)
sample = pd.concat([positives, negatives]).reset_index(drop=True)
print(f"Sample size: {len(sample)}")

Sample size: 200


In [5]:
# ---- 2. Zero-shot prompt (same as Day 4/5, temperature=0.7 this time) ----
def classify_pair(name_a, name_b, temperature=0.7, max_retries=3):
    prompt = f"""You are comparing two product listings to determine if they refer to the same product.

Product A: {name_a}
Product B: {name_b}

Respond with only one word: MATCH or NO_MATCH."""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=5
            )
            return response.choices[0].message.content.strip(), None
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
                continue
            return None, str(e)

def parse_response(raw_response):
    if raw_response is None:
        return None
    r = raw_response.upper()
    if r == "MATCH":
        return 1
    elif r == "NO_MATCH":
        return 0
    elif "NO_MATCH" in r:
        return 0
    elif "MATCH" in r:
        return 1
    return None

In [6]:
# ---- 3. Run 5 times at temperature=0.7 ----
N_RUNS = 5
all_runs = {f'run{i+1}_pred': [] for i in range(N_RUNS)}
all_runs_raw = {f'run{i+1}_raw': [] for i in range(N_RUNS)}

for run_idx in range(N_RUNS):
    print(f"\n--- Run {run_idx + 1}/{N_RUNS} ---")
    for _, row in tqdm(sample.iterrows(), total=len(sample)):
        raw, err = classify_pair(row['name_abt'], row['name_buy'], temperature=0.7)
        pred = parse_response(raw)
        all_runs[f'run{run_idx+1}_pred'].append(pred)
        all_runs_raw[f'run{run_idx+1}_raw'].append(raw)


--- Run 1/5 ---


100%|██████████| 200/200 [02:33<00:00,  1.30it/s]



--- Run 2/5 ---


100%|██████████| 200/200 [02:30<00:00,  1.33it/s]



--- Run 3/5 ---


100%|██████████| 200/200 [02:25<00:00,  1.38it/s]



--- Run 4/5 ---


100%|██████████| 200/200 [02:28<00:00,  1.35it/s]



--- Run 5/5 ---


100%|██████████| 200/200 [02:34<00:00,  1.29it/s]


In [7]:
# ---- 4. Assemble results ----
results = sample[['id_abt', 'id_buy', 'name_abt', 'name_buy', 'label']].copy().reset_index(drop=True)
for k, v in all_runs.items():
    results[k] = v

pred_cols = [f'run{i+1}_pred' for i in range(N_RUNS)]

In [8]:
# ---- 5. Per-pair agreement rate ----
def agreement_rate(row):
    preds = [row[c] for c in pred_cols if pd.notna(row[c])]
    if len(preds) == 0:
        return None
    most_common_count = max(preds.count(0), preds.count(1))
    return most_common_count / len(preds)

results['agreement_rate'] = results.apply(agreement_rate, axis=1)

In [9]:
# ---- 6. Majority vote prediction ----
def majority_vote(row):
    preds = [row[c] for c in pred_cols if pd.notna(row[c])]
    if len(preds) == 0:
        return None
    return 1 if preds.count(1) > preds.count(0) else 0

results['majority_pred'] = results.apply(majority_vote, axis=1)

In [10]:
# ---- 7. Save ----
results.to_csv('../data/processed/abt_buy_consistency_temp07.csv', index=False)
print("\nSaved to data/processed/abt_buy_consistency_temp07.csv")



Saved to data/processed/abt_buy_consistency_temp07.csv


In [11]:
# ---- 8. Summary stats ----
overall_agreement = results['agreement_rate'].mean()
fully_consistent = (results['agreement_rate'] == 1.0).mean()

print(f"\nAverage agreement rate across 200 pairs: {overall_agreement:.3f}")
print(f"Fraction of pairs with 5/5 identical answers: {fully_consistent:.3f}")
print(f"Fraction of pairs with ANY disagreement: {1 - fully_consistent:.3f}")



Average agreement rate across 200 pairs: 0.997
Fraction of pairs with 5/5 identical answers: 0.990
Fraction of pairs with ANY disagreement: 0.010


In [13]:
# ---- 9. Majority vote accuracy vs temp=0 comparison ----
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

valid_majority = results.dropna(subset=['majority_pred'])
p_maj = precision_score(valid_majority['label'], valid_majority['majority_pred'])
r_maj = recall_score(valid_majority['label'], valid_majority['majority_pred'])
f1_maj = f1_score(valid_majority['label'], valid_majority['majority_pred'])
acc_maj = accuracy_score(valid_majority['label'], valid_majority['majority_pred'])

print(f"\nMajority-vote (temp=0.7, 5 runs) on this 200-pair sample:")
print(f"Precision: {p_maj:.3f}, Recall: {r_maj:.3f}, F1: {f1_maj:.3f}, Accuracy: {acc_maj:.3f}")

# Also compute single-run temp=0 accuracy on THIS SAME 200-pair sample for fair comparison
# (re-using your original zero-shot predictions, filtered to this sample)
original_zeroshot = pd.read_csv('../data/processed/abt_buy_llm_zeroshot_preds.csv')
comparable = original_zeroshot.merge(
    sample[['id_abt', 'id_buy']], on=['id_abt', 'id_buy']
)
p0 = precision_score(comparable['label'], comparable['pred_label'])
r0 = recall_score(comparable['label'], comparable['pred_label'])
f10 = f1_score(comparable['label'], comparable['pred_label'])
acc0 = accuracy_score(comparable['label'], comparable['pred_label'])

print(f"\nOriginal temp=0 zero-shot on THIS SAME 200-pair sample:")
print(f"Precision: {p0:.3f}, Recall: {r0:.3f}, F1: {f10:.3f}, Accuracy: {acc0:.3f}")


Majority-vote (temp=0.7, 5 runs) on this 200-pair sample:
Precision: 1.000, Recall: 0.950, F1: 0.974, Accuracy: 0.975

Original temp=0 zero-shot on THIS SAME 200-pair sample:
Precision: 1.000, Recall: 0.940, F1: 0.969, Accuracy: 0.970
